
<span style="color:red">**Please don't work in the original notebook in the DeYonker-lab-scripts repo! Copy to your own QM dir and use your local version**</span>

# How to use this notebook

The first time you open this notebook, you have to:
- make sure you've copied it to your QM dir and you're using that local version, **not** the original in ~/github/DeYonker-lab-scripts/dwappett
- uncomment one of the path lines in the code block below so that the notebook knows where to read your data from

Then each time you use it, all you need to do is:
- in the code block below, select the directories to read and plot the data from by setting fdir as an empty list or list of strings
    - `fdir = []` --> all results from all fxxxxx-fxxxxx directories will be plotted
    - `fdir = ['fxxxxx-fxxxxx',...]` --> only results from the given 'fxxxxx-fxxxxx' directories will be included
    - directories selected in the fdir list (if not left empty) need to contain a [dir]_results.csv file created by collect_CM_results.py otherwise you'll get errors when it tries to read the data!
- click "Run All" to update all plots with the current results from the selected directories

In [1]:
""" specify QMdir path by uncommenting one of these lines """
#path = '/project/ndyonker/chem/cm-MD-processing/MD-QM/QM-A'
#path = '/project/pssntngr/chem/chorismate_mutase/QM-B'
#path = '/project/hksasi/chem/chorismate_mutase/QM-C'
#path = '/project/dwappett/chorismate_mutase/QM-batch-models'

""" specify fxxxxx-fxxxxx directory/directories to plot results from in list like ['f00101-f00200','f00201-f00300'] """
""" or leave empty to compile results for all your finished models """
fdir = []

**Note:** Most plots in this notebook are followed by a secondary code block that identifies outliers visible in the plot so that you don't have to work out how to find them yourselves.

If the outliers are labeled in the plots based on some rule that I've defined, the code block will print those frame names again in case the plot labels are hard to read. 
In some cases I haven't picked a default cutoff for outliers, so the code block just helps you filter the dataframe to identify any points that stand out to you by picking a cutoff that makes sense for your own data.

# Set up notebook environment


## import packages

In [2]:
import os, os.path, glob
import ast
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.lines import Line2D
import tol_colors as tc

## set default font sizes

The following block sets up some default figure widths and font sizes to make it easier to make nice consistent figures for publication. We don't have to use them from day one but I figured it's nice to already have in here for later.

fig sizes for publications:
- single column: approx 8cm wide
- double column: approx 18cm wide


In [3]:
# set default font sizes
# use 16/18/20 for publication quality figures with widths of 8 or 18
#SMALL_SIZE = 16 
#MEDIUM_SIZE = 18 
#BIGGER_SIZE = 20 

# going to be making smaller plots for initial looks otherwise the figs are huge on a big screen. so halving the font sizes to match
SMALL_SIZE = 12
MEDIUM_SIZE = 12 
BIGGER_SIZE = 16 

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

# single column figure width
scolw = 8
# double column figure width
dcolw = 18

## read in data

In [4]:
# get names of results files to check
# just ones in fdir directories if list given otherwise all directories
if fdir:
    QMfiles = [glob.glob(f'{path}/{f}/*_results.csv')[0] for f in fdir]
else:
    QMfiles = glob.glob(f'{path}/*/*_results.csv')
QMfiles.sort()

# read in the data from the selected files
for f in QMfiles:
    if QMfiles.index(f)==0:
        df = pd.read_csv(f,index_col='frame')
    else:
        df = pd.concat([df,pd.read_csv(f,index_col='frame')])

# also read in the model functional groups tables
if fdir:
    FGfiles = [glob.glob(f'{path}/{f}/*_model_FGs.csv')[0] for f in fdir]
else:
    FGfiles = glob.glob(f'{path}/*/*_model_FGs.csv')
FGfiles.sort()
for f in FGfiles:
    if FGfiles.index(f)==0:
        fg_df = pd.read_csv(f,index_col='frame')
    else:
        fg_df = pd.concat([fg_df,pd.read_csv(f,index_col='frame')])

# add FG counts to main dataframe
df['nFG_all'] = fg_df.sum(axis=1)
df['nFG_prot'] = fg_df[[c for c in fg_df.columns if 'WAT' not in c]].sum(axis=1)
df['nFG_SC'] = fg_df[[c for c in fg_df.columns if 'SC' in c]].sum(axis=1)
df['nFG_MC'] = fg_df[[c for c in fg_df.columns if 'MC' in c]].sum(axis=1)
df['nFG_wat'] = fg_df[[c for c in fg_df.columns if 'WAT' in c]].sum(axis=1)

# properly convert lists of imag modes from strings and make column of just max imag mode vals
# also turn nimag lists into just integer val lists and get list that's total no. of irc imag modes
for struc in ['ts','r','p']:
    imagmodes = list(df[f'{struc}_imagmodes'])
    imagmodes = [ast.literal_eval(i) for i in imagmodes]
    df[f'{struc}_imagmodes'] = imagmodes
    df[f'{struc}_maximagmode'] = [i[0] if i else np.nan for i in imagmodes]
    nimag = list(df[f'{struc}_Nimag'])
    df[f'{struc}_Nimag'] = [int(i.replace('Nimag=','')) if isinstance(i,str) else i for i in nimag]
df['tot_irc_Nimag'] = df['r_Nimag']+df['p_Nimag']

## define label/max/min for plotted quantities

In [5]:
# define labels, max and min values so that we can easily set axes titles/legend values and axes limits consistently in plots
# keys of this dict are the column labels from the csv file/dataframes

plotopts = {'dGa': {'longlabel': r'$\Delta G ^\ddagger$ (kcal/mol)',
                'shortlabel': r'$\Delta G ^\ddagger$',
                'max': df['dGa'].max(), 
                'min': df['dGa'].min()
                },
        'dGr': {'longlabel': r'$\Delta G _{rxn}$ (kcal/mol)',
                'shortlabel': r'$\Delta G _{rxn}$',
                'max': df['dGr'].max(), 
                'min': df['dGr'].min()
                },
        'size': {'longlabel': 'model size (no. atoms)',
                'shortlabel': 'size',
                'max': df['size'].max(), 
                'min': df['size'].min()
                },
        'charge': {'longlabel': 'overall charge',
                'shortlabel': 'charge',
                'max': df['charge'].max(), 
                'min': df['charge'].min()
                },
        'fnum': {'longlabel': 'model frame number',
                'shortlabel': 'model',
                'max': df['fnum'].max(), 
                'min': df['fnum'].min()
                },
        'nFG_all': {'longlabel': 'no. FGs in model',
                'shortlabel': 'no. FGs',
                'max': df['nFG_all'].max(), 
                'min': df['nFG_all'].min()
                },
        'nFG_prot': {'longlabel': 'no. protein FGs in model',
                'shortlabel': 'no. protein FGs',
                'max': df['nFG_prot'].max(), 
                'min': df['nFG_prot'].min()
                },
        'nFG_SC': {'longlabel': 'no. SC fragments in model',
                'shortlabel': 'no. SCs',
                'max': df['nFG_SC'].max(), 
                'min': df['nFG_SC'].min()
                },
        'nFG_MC': {'longlabel': 'no. MC fragments in model',
                'shortlabel': 'no. MCs',
                'max': df['nFG_MC'].max(), 
                'min': df['nFG_MC'].min()
                },
        'nFG_wat': {'longlabel': 'no. waters in model',
                'shortlabel': 'no. waters',
                'max': df['nFG_wat'].max(), 
                'min': df['nFG_wat'].min()
                }
        }

# Structure checks

## max dists moved during opts

plotting difference between max heavy atom movement and max hydrogen movement should hopefully help identify models where hydrogens have detached/transferred during the opts and messed up the results!

in the plot below, models are labeled if the maxmove value is >6 or the magnitude of the diff is >2

In [ ]:
# get max and min of all maxmove cols so that all axes can have same limits
maxmovecols = [c for c in list(df.columns.values) if c.startswith('maxmove')]
allmin = min([np.nanmin(df[i]) for i in maxmovecols])
allmax = max([np.nanmax(df[i]) for i in maxmovecols])

# create lists of struc pairs that match the col names and clearer names for plot titles
titlelabels = ['initialopt vs template','tsopt vs tsguess','reactant vs tsopt','product vs tsopt']
pairs = ['tmp-init','guess-ts','ts-r','ts-p']

# get min and max diff also for axes limits
diffmax = max([np.nanmax(df[f'maxmove_H_{pair}']-df[f'maxmove_heavy_{pair}']) for pair in pairs])
diffmin = min([np.nanmin(df[f'maxmove_H_{pair}']-df[f'maxmove_heavy_{pair}']) for pair in pairs])

# define markers that will be used so it's easier to make one combined legend
legend_elements = [Line2D([0], [0], marker='o', ls='none', label='heavy atoms', color=tc.bright.blue, markersize=10), 
Line2D([0], [0], marker='s', ls='none', label='H atoms', color=tc.bright.red, markersize=10),
Line2D([0], [0], marker='^', ls='none', label='H-heavy diff.', color=tc.bright.green, markersize=10)]

# set up dict to collect model numbers of the outliers
outliermodels = {}
for pair in pairs:
    for t in ['H','heavy','diff']:
        outliermodels[(pair,t)] = []

fig, axs = plt.subplots(nrows=2,ncols=4,figsize=(14,6),layout='constrained')
for i,pair in enumerate(pairs):
    # top row of plots: actual values
    df.plot(x='fnum',y=f'maxmove_heavy_{pair}',kind='scatter',ax=axs[0,i],color=tc.bright.blue,marker='o',label='heavy',legend=None)
    df.plot(x='fnum',y=f'maxmove_H_{pair}',kind='scatter',ax=axs[0,i],color=tc.bright.red,marker='s', label='H',legend=None)
    axs[0,i].set_ylim(bottom=0,top=allmax+2)
    axs[0,i].set_xlim(left=plotopts['fnum']['min']-1,right=plotopts['fnum']['max']+1)
    axs[0,i].set_xlabel(None)
    axs[0,i].set_ylabel(None)
    axs[0,i].set_title(titlelabels[i])
    # bottom row of plots: differences
    axs[1,i].axhline(y=0,color='black',linewidth=1)
    axs[1,i].scatter(x=df['fnum'],y=df[f'maxmove_H_{pair}']-df[f'maxmove_heavy_{pair}'],color=tc.bright.green,marker='^')
    axs[1,i].set_ylim(bottom=diffmin-1,top=diffmax+1)
    axs[1,i].set_xlim(left=plotopts['fnum']['min']-1,right=plotopts['fnum']['max']+1)
    axs[1,i].set_xlabel(None)
    axs[1,i].set_ylabel(None)
    # label points where maxmove H/heavy are >6 (only one label per model even if both are big) and diff > 2
    for model in list(df.index.values):
        if df.loc[model,f'maxmove_H_{pair}']>6:
            axs[0,i].annotate(df.loc[model,'fnum'],(df.loc[model,'fnum'],df.loc[model,f'maxmove_H_{pair}']),size='x-small')
            outliermodels[(pair,'H')].append(model)
        if df.loc[model,f'maxmove_heavy_{pair}']>6:
            axs[0,i].annotate(df.loc[model,'fnum'],(df.loc[model,'fnum'],df.loc[model,f'maxmove_heavy_{pair}']),size='x-small')
            outliermodels[(pair,'heavy')].append(model)
        mmdiff = df.loc[model,f'maxmove_H_{pair}'] - df.loc[model,f'maxmove_heavy_{pair}']
        if mmdiff > 2 or mmdiff < -2:
            axs[1,i].annotate(df.loc[model,'fnum'],(df.loc[model,'fnum'],mmdiff),size='x-small')
            outliermodels[(pair,'diff')].append(model)

fig.legend(handles=legend_elements,loc='outside upper center',ncols=3,handletextpad=0.1,columnspacing=2)

axs[0,0].set_ylabel('max dist moved by\nany atom during opt '+r'($\AA$)')
axs[1,0].set_ylabel('diff. between max H movement\nand max heavy atom movement')
fig.supxlabel(plotopts['fnum']['longlabel'])
fig.align_labels()


In [ ]:
### show outliers in case it's hard to read from plot ###
### filtering out any combinations in dict that don't have any outliers ###
{key: outliermodels[key] for key in outliermodels.keys() if outliermodels[key]}

## RMSDs of strucs before/after opts

In [ ]:
rmscols = [c for c in df.columns if c.startswith('rms')]
allmin = min([np.nanmin(df[i]) for i in rmscols])
allmax = max([np.nanmax(df[i]) for i in rmscols])

titlelabels = ['initialopt vs template','tsopt vs tsguess','reactant vs tsopt','product vs tsopt','product vs reactant']
pairs = ['tmp-init','guess-ts','ts-r','ts-p','p-r']
markers = ['o','s','^']
cols = [tc.high_contrast.yellow,tc.high_contrast.red,tc.high_contrast.blue]

fig, axs = plt.subplots(nrows=3,ncols=5,figsize=(14,7),layout='constrained')
for i,pair in enumerate(pairs):
    for j,rmsd in enumerate(['all','prot','wat']):
        # top row of plots: actual values
        df.plot(x='fnum',y=f'rms_{pair}_{rmsd}',kind='scatter',ax=axs[j,i],color=cols[j],marker=markers[j])
        axs[j,i].set_ylim(bottom=0,top=allmax+0.2)
        axs[j,i].set_xlim(left=plotopts['fnum']['min']-1,right=plotopts['fnum']['max']+1)
        axs[j,i].set_xlabel(None)
        axs[j,i].set_ylabel(None)
        if j==0:
            axs[j,i].set_title(titlelabels[i])
        if i==0:
            axs[j,i].set_ylabel(f'RMSD ({rmsd})')


fig.supxlabel(plotopts['fnum']['longlabel'])
fig.align_labels()



In [ ]:
### quick code to filter df to find outliers ###
### change the vals of these three variables as needed ###
filt = 1.5        # set rmsd cutoff for outliers
pair = 'ts-p'   # select pair out of ['tmp-init','guess-ts','ts-r','ts-p','p-r']
rmsd = 'wat'    # select rmsd type out of ['all','prot','wat']

df.loc[df[f'rms_{pair}_{rmsd}']>filt,f'rms_{pair}_{rmsd}']


## ligand C1-C9 and C5-O7 bond distances

should be a bond between C5 and O7 in reactant and a bond between C1 and C9 in product. distances should be similar in ts.

In [ ]:
allmin = min([np.nanmin(df[i]) for i in ['ts_C5-O7_dist','r_C5-O7_dist','p_C5-O7_dist','ts_C1-C9_dist','r_C1-C9_dist','p_C1-C9_dist']])
allmax = max([np.nanmax(df[i]) for i in ['ts_C5-O7_dist','r_C5-O7_dist','p_C5-O7_dist','ts_C1-C9_dist','r_C1-C9_dist','p_C1-C9_dist']])

fig, axs = plt.subplots(nrows=3,ncols=1,figsize=(10,6),layout='constrained')
for i, ydata in enumerate(['ts','r','p']):
    df.plot(x='fnum',y=f'{ydata}_C5-O7_dist',kind='scatter',ax=axs[i],color=tc.bright.blue,marker='o',label='ligand C5-O7 dist (bond in reactant)')
    df.plot(x='fnum',y=f'{ydata}_C1-C9_dist',kind='scatter',ax=axs[i],color=tc.bright.red,marker='s',label='ligand C1-C9 dist (bond in product)')
    axs[i].set_ylim(bottom=0.8,top=allmax+0.2)
    axs[i].set_xlim(left=plotopts['fnum']['min']-1,right=plotopts['fnum']['max']+1)
    axs[i].set_xlabel(None)
    axs[i].set_ylabel(ydata)
    axs[i].get_legend().remove()

fig.legend(['ligand C5-O7 dist (bond in reactant)','ligand C1-C9 dist (bond in product)'],ncols=2,loc='outside upper center',markerscale=2,handletextpad=0.1,columnspacing=2)
fig.supylabel(r'distance ($\AA$)')
fig.supxlabel(plotopts['fnum']['longlabel'])
fig.align_labels()

In [ ]:
### quick code to filter df to find outliers ###
### change the vals of these three variables as needed ###
filt = 4        # set distance cutoff to determine what to show
struc = 'p'     # 'ts' or 'r' or 'p'
bond = 'C5-O7'  # 'C5-O7' or 'C1-C9'

df.loc[df[f'{struc}_{bond}_dist']>filt,f'{struc}_{bond}_dist']

## ligand dihedrals around the p/r bonds

if conformation of ligand is similar to ts, dihedral should be around 0. big values indicate that the reactant/product is twisting around the relevant bond into an alternative shape?

In [ ]:
allmin = min([np.nanmin(df[i]) for i in ['r_C1-C5-O7-C9_dihedral','p_C5-C1-C9-O7_dihedral']])
allmax = max([np.nanmax(df[i]) for i in ['r_C1-C5-O7-C9_dihedral','p_C5-C1-C9-O7_dihedral']])

fig, axs = plt.subplots(nrows=2,ncols=1,figsize=(10,5),layout='constrained')
df.plot(x='fnum',y='r_C1-C5-O7-C9_dihedral',kind='scatter',ax=axs[0],color=tc.bright.blue,marker='o')
axs[0].set_title('reactant C1-C5-O7-C9 dihedral')
df.plot(x='fnum',y='p_C5-C1-C9-O7_dihedral',kind='scatter',ax=axs[1],color=tc.bright.red,marker='s')
axs[1].set_title('product C5-C1-C9-O7 dihedral')

for i in [0,1]:
    axs[i].set_ylim(bottom=allmin-5,top=allmax+5)
    axs[i].set_xlim(left=plotopts['fnum']['min']-1,right=plotopts['fnum']['max']+1)
    axs[i].set_xlabel(None)
    axs[i].set_ylabel(None)
    axs[i].axhline(y=0,color='black',linewidth=1)

fig.supylabel(r'dihedral angle $\phi$')
fig.supxlabel(plotopts['fnum']['longlabel'])
fig.align_labels()

In [ ]:
### quick code to filter df to find outliers ###
### change the vals of these two variables as needed ###
filt = 50                           # set cutoff to determine what to show, checks +/- so just give positive val
dihed = 'r_C1-C5-O7-C9_dihedral'    # 'r_C1-C5-O7-C9_dihedral' or 'p_C5-C1-C9-O7_dihedral'

df.loc[(df[dihed]>filt)|(df[dihed]<-filt),dihed]

# Imaginary mode checks

## ts imaginary mode

In [ ]:
fig, axs = plt.subplots(nrows=1,ncols=1,figsize=(10,3),layout='constrained')
df.plot(x='fnum',y='ts_maximagmode',kind='scatter',ax=axs,color=tc.bright.blue,marker='o')
axs.set_ylim(bottom=np.nanmin(df['ts_maximagmode'])-20,top=np.nanmax(df['ts_maximagmode'])+20)
axs.set_xlim(left=plotopts['fnum']['min']-1,right=plotopts['fnum']['max']+1)
axs.set_xlabel(plotopts['fnum']['longlabel'])
axs.set_ylabel(r'TS imaginary mode ($cm^{-1}$)')


In [ ]:
### quick code to filter df to find outliers ###
### change the vals of the filt variable as needed ###
filt = -200     # set cutoff to determine what to show

df.loc[df['ts_maximagmode']>filt,'ts_maximagmode']

## irc imaginary modes

In [ ]:
fig, axs = plt.subplots(nrows=2,ncols=2,figsize=(10,6),layout='constrained')
df.plot(ax=axs[0,0],x='fnum',y='r_Nimag',kind='scatter',color=tc.bright.blue,marker='o')
df.plot(ax=axs[1,0],x='fnum',y='p_Nimag',kind='scatter',color=tc.bright.blue,marker='o')
df.plot(ax=axs[0,1],x='fnum',y='r_maximagmode',kind='scatter',color=tc.bright.blue,marker='o')
df.plot(ax=axs[1,1],x='fnum',y='p_maximagmode',kind='scatter',color=tc.bright.blue,marker='o')


# Model contents

## model sizes, charges, number of fragments (all/SC/MC/wat)

In [ ]:
colours = [tc.bright.green,tc.bright.yellow,tc.bright.blue,tc.bright.cyan,tc.bright.red,tc.bright.purple]
#[tc.bright.green,tc.medium_contrast.dark_blue,tc.bright.yellow,tc.medium_contrast.dark_red]
markers = ['s','o','d','^']

fig, axs = plt.subplots(nrows=3,ncols=4,figsize=(12,6),layout='constrained')
axs = axs.reshape(-1)
for i, ydata in enumerate(['size','charge','nFG_all','nFG_wat','nFG_SC','nFG_MC']):
    df.plot(x='fnum',y=ydata,kind='scatter',ax=axs[2*i],color=colours[i],marker='o')
    axs[2*i].set_xlabel(plotopts['fnum']['shortlabel'])
    axs[2*i].set_ylabel(plotopts[ydata]['shortlabel'])
    binrange = np.arange(np.floor(plotopts[ydata]['min'])-0.5, np.ceil(plotopts[ydata]['max'])+1.5,1)
    df.hist(column=ydata,
            ax=axs[2*i+1],
            bins=binrange,
            rwidth=0.9,
            legend=False, 
            grid=False,
            color=colours[i],
            orientation='horizontal')
    axs[2*i+1].set_ylabel(plotopts[ydata]['shortlabel'])
    axs[2*i+1].set_xlabel('no. models')
    axs[2*i+1].set_title(None)
    # make y axis only show integer ticks (skipping size/no.FGs because doing this makes too many yticks and they already only show )
    if ydata not in ['size','nFG_all']:
        axs[2*i].yaxis.set_major_locator(MaxNLocator(integer=True))
        axs[2*i+1].yaxis.set_major_locator(MaxNLocator(integer=True))
    #make scatter ylims match histogram ylims
    axs[2*i].set_ylim(axs[2*i+1].get_ylim())


fig.supxlabel(plotopts['fnum']['longlabel'])
fig.align_labels()


# Free energy results

## free energies vs fnum/size/charge

In [ ]:
colours = [tc.medium_contrast.dark_blue,tc.medium_contrast.dark_red]
markers = ['o','^']

ci_act = [df['dGa'].mean()-2*df['dGa'].std(ddof=0),df['dGa'].mean()+2*df['dGa'].std(ddof=0)]
ci_rxn = [df['dGr'].mean()-2*df['dGr'].std(ddof=0),df['dGr'].mean()+2*df['dGr'].std(ddof=0)]

outliermodels = {'dGa': [], 'dGr': []}

fig, axs = plt.subplots(nrows=2,ncols=3,figsize=(12,5.5),layout='constrained')

for i, ydata in enumerate(['dGa','dGr']):
    for j, xdata in enumerate(['fnum','size','charge']):
        axs[i,j].axhline(y=df[ydata].mean(),color='black',linestyle='--',linewidth=1)
        df.plot(x=xdata,y=ydata,kind='scatter',ax=axs[i,j],color=colours[i],marker=markers[i])
        axs[i,j].set_ylim(bottom=plotopts[ydata]['min']-1,top=plotopts[ydata]['max']+1)
        axs[i,j].set_xlabel(None)
        axs[i,j].set_ylabel(None)
        if xdata == 'charge':
            axs[i,j].xaxis.set_major_locator(MaxNLocator(integer=True))
            axs[i,j].set_xlim(left=plotopts[xdata]['min']-0.5,right=plotopts[xdata]['max']+0.5)
        else:
            axs[i,j].set_xlim(left=plotopts[xdata]['min']-2,right=plotopts[xdata]['max']+2)
        # add labels to outliers
        if ydata == 'dGa':
            for model in df.index.values:
                if df.loc[model,'dGa'] < ci_act[0] or df.loc[model,'dGa'] > ci_act[1]:
                    axs[i,j].annotate(df.loc[model,'fnum'],(df.loc[model,xdata],df.loc[model,'dGa']),size='x-small')
                    outliermodels['dGa'].append(model)
        elif ydata == 'dGr':
            for model in df.index.values:
                if df.loc[model,'dGr'] < ci_rxn[0] or df.loc[model,'dGr'] > ci_rxn[1]:
                    axs[i,j].annotate(df.loc[model,'fnum'],(df.loc[model,xdata],df.loc[model,'dGr']),size='x-small')
                    outliermodels['dGr'].append(model)
        if i==1:
            axs[i,j].set_xlabel(plotopts[xdata]['longlabel'])


axs[0,0].set_ylabel(plotopts['dGa']['longlabel'])
axs[1,0].set_ylabel(plotopts['dGr']['longlabel'])

fig.align_labels()
fig.suptitle(f'mean {plotopts["dGa"]["shortlabel"]} = {round(df["dGa"].mean(),1)} kcal/mol; mean {plotopts["dGr"]["shortlabel"]} = {round(df["dGr"].mean(),1)} kcal/mol',size=MEDIUM_SIZE)


In [ ]:
### show outliers in case it's hard to read from plot ###
# converting dict contents list>set>list removes duplicates
{key: list(set(outliermodels[key])) for key in outliermodels.keys()}

## free energies vs no. fragments

outliers are labeled in plots above so not repeating them here

In [ ]:
colours = [tc.medium_contrast.dark_blue,tc.medium_contrast.dark_red]
markers = ['o','^']

fig, axs = plt.subplots(nrows=2,ncols=5,figsize=(12,5),layout='constrained')

for i, ydata in enumerate(['dGa','dGr']):
    for j, xdata in enumerate(['nFG_all','nFG_wat','nFG_prot','nFG_SC','nFG_MC']):
        axs[i,j].axhline(y=0,color='black',linewidth=1)
        df.plot(x=xdata,y=ydata,kind='scatter',ax=axs[i,j],color=colours[i],marker=markers[i])
        if j==0:
            axs[i,j].set_ylim(bottom=plotopts[ydata]['min']-1,top=plotopts[ydata]['max']+1)
        else:
            axs[i,j].sharey(axs[i,0])
        axs[i,j].set_xlabel(None)
        axs[i,j].set_ylabel(None)
        axs[i,j].xaxis.set_major_locator(MaxNLocator(integer=True))
        axs[i,j].set_xlim(left=plotopts[xdata]['min']-0.5,right=plotopts[xdata]['max']+0.5)
        if i==1:
            axs[i,j].set_xlabel(plotopts[xdata]['shortlabel'])
            axs[0,j].sharex(axs[1,j])

# I dunno why sharing the axes doesn't remove the tick labels from all subplots but some have to be manually fixed up
axs[1,4].set_yticklabels([])
axs[0,4].set_xticklabels([])

axs[0,0].set_ylabel(plotopts['dGa']['longlabel'])
axs[1,0].set_ylabel(plotopts['dGr']['longlabel'])

fig.align_labels()

## free energies coloured by ts-r and p-r RMSDs

In [ ]:
fig, axs = plt.subplots(nrows=2,ncols=3,figsize=(10,5),layout='constrained')

for i, rmstype in enumerate(['all','prot','wat']):
    p1 = df.plot.scatter(x='fnum',y='dGa',ax=axs[0,i],c=f'rms_ts-r_{rmstype}',cmap='Blues',colorbar=False,ylim=(plotopts['dGa']['min']-1,plotopts['dGa']['max']+1),xlim=(plotopts['fnum']['min']-1,plotopts['fnum']['max']+1))
    p2 = df.plot.scatter(x='fnum',y='dGr',ax=axs[1,i],c=f'rms_p-r_{rmstype}',cmap='Reds',colorbar=False,ylim=(plotopts['dGr']['min']-1,plotopts['dGr']['max']+1),xlim=(plotopts['fnum']['min']-1,plotopts['fnum']['max']+1))
    cbar1 = plt.colorbar(p1.collections[0], ax=axs[0,i], location='top', orientation='horizontal',label=f'ts-r rmsd ({rmstype})')
    cbar2 = plt.colorbar(p2.collections[0], ax=axs[1,i], location='top', orientation='horizontal',label=f'p-r rmsd ({rmstype})')
    axs[0,i].set_ylabel(None)
    axs[1,i].set_ylabel(None)
    axs[0,i].set_xlabel(None)
    axs[1,i].set_xlabel(None)
    
axs[0,0].set_ylabel(plotopts['dGa']['longlabel'])
axs[1,0].set_ylabel(plotopts['dGr']['longlabel'])
fig.supxlabel('frame')

fig.align_labels()

## free energies coloured to show ircs with imag modes

In [ ]:
#df.plot.scatter(x='fnum',y='dGa',c='tot_irc_Nimag',cmap='viridis_r',ylim=(plotopts['dGa']['min']-1,plotopts['dGa']['max']+1),xlim=(plotopts['fnum']['min']-1,plotopts['fnum']['max']+1))

colours = [(tc.medium_contrast.dark_blue,tc.medium_contrast.light_blue),(tc.medium_contrast.dark_red,tc.medium_contrast.light_red)]
fig, axs = plt.subplots(nrows=1,ncols=3,figsize=(12,3),layout='constrained')

# first plot: dGa coloured by r Nimag
df.loc[df['r_Nimag']==0,:].plot(ax=axs[0],kind='scatter',x='fnum',y='dGa',color=tc.medium_contrast.light_blue,marker='o',label='r Nimag=0',
                                xlabel=plotopts['fnum']['longlabel'],ylabel=plotopts['dGa']['longlabel'],
                                ylim=(plotopts['dGa']['min']-1,plotopts['dGa']['max']+1),xlim=(plotopts['fnum']['min']-1,plotopts['fnum']['max']+1))
df.loc[df['r_Nimag']>0,:].plot(ax=axs[0],kind='scatter',x='fnum',y='dGa',color=tc.medium_contrast.dark_blue,marker='o', label='r Nimag>0',
                                xlabel=plotopts['fnum']['longlabel'],ylabel=plotopts['dGa']['longlabel'],
                                ylim=(plotopts['dGa']['min']-1,plotopts['dGa']['max']+1),xlim=(plotopts['fnum']['min']-1,plotopts['fnum']['max']+1))

# second plot: dGr coloured by r Nimag
df.loc[df['r_Nimag']==0,:].plot(ax=axs[1],kind='scatter',x='fnum',y='dGr',color=tc.medium_contrast.light_red,marker='o',label='r Nimag=0',
                                xlabel=plotopts['fnum']['longlabel'],ylabel=plotopts['dGr']['longlabel'],
                                ylim=(plotopts['dGr']['min']-1,plotopts['dGr']['max']+1),xlim=(plotopts['fnum']['min']-1,plotopts['fnum']['max']+1))
df.loc[df['r_Nimag']>0,:].plot(ax=axs[1],kind='scatter',x='fnum',y='dGr',color=tc.medium_contrast.dark_red,marker='o', label='r Nimag>0',
                                xlabel=plotopts['fnum']['longlabel'],ylabel=plotopts['dGr']['longlabel'],
                                ylim=(plotopts['dGr']['min']-1,plotopts['dGr']['max']+1),xlim=(plotopts['fnum']['min']-1,plotopts['fnum']['max']+1))

# third plot: dGr coloured by p Nimag
df.loc[df['p_Nimag']==0,:].plot(ax=axs[2],kind='scatter',x='fnum',y='dGr',color=tc.medium_contrast.light_red,marker='o',label='p Nimag=0',
                                xlabel=plotopts['fnum']['longlabel'],ylabel=plotopts['dGr']['longlabel'],
                                ylim=(plotopts['dGr']['min']-1,plotopts['dGr']['max']+1),xlim=(plotopts['fnum']['min']-1,plotopts['fnum']['max']+1))
df.loc[df['p_Nimag']>0,:].plot(ax=axs[2],kind='scatter',x='fnum',y='dGr',color=tc.medium_contrast.dark_red,marker='o', label='p Nimag>0',
                                xlabel=plotopts['fnum']['longlabel'],ylabel=plotopts['dGr']['longlabel'],
                                ylim=(plotopts['dGr']['min']-1,plotopts['dGr']['max']+1),xlim=(plotopts['fnum']['min']-1,plotopts['fnum']['max']+1))

fig.align_labels()

# extra: details of everyone's active sites

sticking this here like an appendix in case anyone needs it but it's probably not necessary for this notebook


|             | ligand | active site   | frames      | model type | parent directory for project                        |
|-------------|--------|---------------|-------------|------------|-----------------------------------------------------|
| **Dr D.**   | A:128  | A/C interface | all         | individual | /project/ndyonker/chem/cm-MD-processing/MD-QM/QM-A  |
| **Pedro**   | B:256  | B/A interface | all         | individual | /project/pssntngr/chem/chorismate_mutase/QM-B/      |
| **Haritha** | C:384  | C/B interface | all         | individual | /project/hksasi/chem/chorismate_mutase/QM-C/        |
| **Domi**    | all    | all           | every 100th | batch      | /project/dwappett/chorismate_mutase/QM-batch-models |
